# 1st LAB: Word Embeddings

In [1]:
import re
import string

import nltk
import numpy as np
import pandas as pd

# Loading the data

In [2]:
# US presidential inaugural speeches, from the NLTK corpora
nltk.download('inaugural') ###
from nltk.corpus import inaugural

[nltk_data] Downloading package inaugural to /root/nltk_data...
[nltk_data]   Package inaugural is already up-to-date!


In [3]:
# Extract the year, president's name and speech text from each file
data = pd.DataFrame({
    # Extract the year from the first 4 characters of each filename
    'year': [int(f[:4]) for f in inaugural.fileids()],
    # Extract the president's name from 6th char to last before ".txt"
    'president': [f[5:-4] for f in inaugural.fileids()],
    # Retrieve the full text of each inaugural speech
    'text': [inaugural.raw(f) for f in inaugural.fileids()],
}).sort_values('year', ignore_index=True)  # Sort speeches by year

data.head()

,year,president,text
0,1789,Washington,Fellow-Citizens of the Senate and of the House...
1,1793,Washington,"Fellow citizens, I am again called upon by the..."
2,1797,Adams,"When it was first perceived, in early times, t..."
3,1801,Jefferson,Friends and Fellow Citizens:\n\nCalled upon to...
4,1805,Jefferson,"Proceeding, fellow citizens, to that qualifica..."


In [4]:
# Define functions to remove punctuation and numbers from text
def remove_punctuation(text):
    return re.sub(f"[{re.escape(string.punctuation)}]", " ", text)

def remove_numbers(text):
    return re.sub(r"\d+", "", text)

In [5]:
# Convert text to lowercase
data["preprocessed"] = data["text"].str.lower()

# Remove numbers and punctuation
data["preprocessed"] = data["preprocessed"].apply(remove_numbers)
data["preprocessed"] = data["preprocessed"].apply(remove_punctuation)

# Split text into individual words (tokens)
data["tokens"] = data["preprocessed"].str.split()

data.head()

,year,president,text,preprocessed,tokens
0,1789,Washington,Fellow-Citizens of the Senate and of the House...,fellow citizens of the senate and of the house...,"[fellow, citizens, of, the, senate, and, of, t..."
1,1793,Washington,"Fellow citizens, I am again called upon by the...",fellow citizens i am again called upon by the...,"[fellow, citizens, i, am, again, called, upon,..."
2,1797,Adams,"When it was first perceived, in early times, t...",when it was first perceived in early times t...,"[when, it, was, first, perceived, in, early, t..."
3,1801,Jefferson,Friends and Fellow Citizens:\n\nCalled upon to...,friends and fellow citizens \n\ncalled upon to...,"[friends, and, fellow, citizens, called, upon,..."
4,1805,Jefferson,"Proceeding, fellow citizens, to that qualifica...",proceeding fellow citizens to that qualifica...,"[proceeding, fellow, citizens, to, that, quali..."


In [6]:
print(len(data))

60


# Introduction to gensim

Before the exercises, let's see **how to use** the gensim tools on a toy corpus of 5 sentences.

> **BEWARE:** the corpus is far too small, so the results (vectors, similar words) mean nothing. Look only at **which functions** we call and **in which form** we pass the data.

In [7]:
from gensim.models import Word2Vec

# Word2Vec wants a LIST OF SENTENCES, each sentence a LIST OF WORDS
toy = [
    ['i', 'like', 'green', 'eggs', 'and', 'ham'],
    ['green', 'eggs', 'for', 'breakfast'],
    ['i', 'like', 'bacon', 'and', 'toast'],
    ['we', 'eat', 'green', 'eggs', 'with', 'toast'],
    ['the', 'king', 'eats', 'ham', 'and', 'bacon'],
]

# sg=1: skip-gram, sg=0: CBOW
# min_count=1: keep also the words seen only once (the default, 5, would remove all of them here)
toy_model = Word2Vec(sentences=toy, vector_size=10, window=2, min_count=1, sg=1, seed=42)

print('Vocabulary:', list(toy_model.wv.key_to_index))
print('Vector of eggs:', toy_model.wv['eggs'].round(2))
print('Most similar to eggs:', toy_model.wv.most_similar('eggs', topn=3))

# word analogies: positive words are added, negative words are subtracted (eggs + bacon - ham)
print('eggs + bacon - ham:', toy_model.wv.most_similar(positive=['eggs', 'bacon'], negative=['ham'], topn=3))

# a word that is not in the corpus has no vector
print("Is 'pizza' in the vocabulary?", 'pizza' in toy_model.wv)

Vocabulary: ['and', 'eggs', 'green', 'toast', 'bacon', 'ham', 'like', 'i', 'eats', 'king', 'the', 'with', 'eat', 'we', 'breakfast', 'for']
Vector of eggs: [ 0.01  0.1   0.05  0.05  0.04  0.06  0.   -0.07  0.07 -0.01]
Most similar to eggs: [('king', 0.3018323481082916), ('with', 0.2954557240009308), ('eats', 0.2678139805793762)]
eggs + bacon - ham: [('and', 0.6240828037261963), ('eat', 0.36743733286857605), ('with', 0.24361400306224823)]
Is 'pizza' in the vocabulary? False


### Bigrams with `Phrases`

`Phrases` finds the pairs of words that often appear together and joins them with `_`.

In [8]:
from gensim.models import Phrases

# min_count and threshold are very low only because the toy corpus is tiny: on real data keep the defaults
toy_phrases = Phrases(toy, min_count=1, threshold=1)

# apply it to the sentences: the result is a new corpus, in the same form (list of lists of words)
toy_bigrams = list(toy_phrases[toy])
print(toy_bigrams)

[['i_like', 'green_eggs', 'and', 'ham'], ['green_eggs', 'for', 'breakfast'], ['i_like', 'bacon', 'and', 'toast'], ['we', 'eat', 'green_eggs', 'with', 'toast'], ['the', 'king', 'eats', 'ham', 'and', 'bacon']]


### Document vectors with `Doc2Vec`

Many tasks work on **whole documents** (sentiment analysis, passage retrieval, clustering), so we need one vector per document.

- **First idea:** average the word vectors of the document, or better, weight them with TF-IDF (this is Exercise 3.3).
- **Doc2Vec** (Le & Mikolov, 2014): each document gets **its own vector**, learned together with the word vectors. It usually works better than averaging.

**How Doc2Vec learns.** Each document is treated as **one extra word**, a tag like `doc 1`, and Word2Vec is trained as usual:

Document 1: `"fry the eggs and the bacon"`

| | tag | context words | word to guess |
|---|---|---|---|
| window 1 | `doc 1` | `fry` `the` | `eggs` |
| window 2 | `doc 1` | `eggs` `and` | `the` |
| window 3 | `doc 1` | `and` `the` | `bacon` |

- The tag is added to **every window** of its document: `doc 1` sees all the words of document 1, so its vector absorbs the **topic** of the whole document (here: near `eggs`, `bacon`, `fry`).
- A **new document** (not seen in training): the word vectors stay fixed and only its tag is learned. This is what `infer_vector` does.

In gensim each document is a `TaggedDocument`: the list of its words + a tag (here, its number). After training, `model.dv[i]` is the vector of document `i`.

In [9]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

toy_docs = [TaggedDocument(words, [i]) for i, words in enumerate(toy)]
print(toy_docs[0])

toy_doc_model = Doc2Vec(toy_docs, vector_size=10, window=2, min_count=1, seed=42)

# the vector learned for a document of the corpus, by its tag
print('Vector of document 0:', toy_doc_model.dv[0].round(2))

# the vector of a NEW text (a list of words, not a string)
print('Vector of a new text:', toy_doc_model.infer_vector(['green', 'eggs', 'and', 'bacon']).round(2))

TaggedDocument<['i', 'like', 'green', 'eggs', 'and', 'ham'], [0]>
Vector of document 0: [ 0.08  0.09  0.08 -0.08 -0.08 -0.05 -0.04  0.05 -0.07 -0.08]
Vector of a new text: [ 0.01  0.04 -0.03  0.03 -0.04  0.04 -0.04  0.03  0.03 -0.04]


# Exercise 3.1: Static Word Embeddings

---




Steps:

Part 1 **Training Word2vec on a dataset**:
- Train a Word2Vec model on the presidents' speeches. You must pass to Word2Vec a list of lists of tokens (a list of tokenized sentences). Use CBOW and a window of 5. Documentation: https://radimrehurek.com/gensim/models/word2vec.html#gensim.models.word2vec.Word2Vec
- Find the words most similar to, e.g. "state", "states", "president", or others.
- Compare the effectiveness of CBOW and Skipgram. Which one works better?


Part 2 **Using pre-trained static models**:
- Load a pre-trained gensim model (available models: https://github.com/RaRe-Technologies/gensim-data#Models) (Choose a small one to avoid waiting for the download, e.g. `glove-wiki-gigaword-100`).
- Evaluate the following relations:
  - king + woman - man = ?
  - France - Paris + London = ?
  - Find out some for yourselves

**Question: Find the words most similar to "italy" using the 2 models (trained and pretrained). What do you observe? Why? Any solution?**



In [10]:
nltk.download('punkt') ###< run once
nltk.download('punkt_tab') ###< run once, needed by recent NLTK versions

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

**Data preparation (given).** Word2Vec learns from the words *around* each word, so it needs **one list of words per sentence**, not per speech. The next cells build it: `corpus` is the input for Word2Vec, and you will use it also in Exercises 3.2 and 3.3.

In [11]:
data['sent_tokens'] = data.text.str.lower()

data.sent_tokens = data.sent_tokens.str.replace(r'\d+', ' ', regex=True)
data.sent_tokens = data.sent_tokens.apply(lambda input_text : [remove_punctuation(t).split() for t in nltk.sent_tokenize(input_text)])

In [12]:
print('Number of sentences:', len(data.sent_tokens.sum()))

Number of sentences: 5360


In [13]:
corpus = data.sent_tokens.sum() # a list of sentences (each one a list of words), instead of a list of speeches
print(corpus[0][:10])

['fellow', 'citizens', 'of', 'the', 'senate', 'and', 'of', 'the', 'house', 'of']


In [14]:
# Your code here

from gensim import downloader
from gensim.models import Word2Vec

> **Note: your results may differ.** `seed=42` fixes the starting point, but gensim trains with several threads in parallel, and the order in which the threads work changes at every run. So the similar words (and their scores) can change a little from one run to another, and from the ones of your classmates. The same holds for `Doc2Vec` in Exercise 3.4.
>
> Look at the *kind* of words you get, not at the exact list.

# Exercise 3.2: Embeddings with multiword ngrams

---



**Training Word2vec on a dataset**:
- Train a bigram detector using gensim.
- Train a Word2Vec model on the presidents' speeches. You must pass to Word2Vec a list of lists of bigram tokens (a list of tokenized sentences).
- Find the words most similar to, e.g. "united_states", "american_people" or others.
- Try both word2vec architectures.

**Have you seen the sparsity problem?**

*Hint: see `Phrases` in the Introduction to gensim, and apply it to `corpus` (Exercise 3.1). On the real corpus keep the default `min_count` and `threshold`. To count how many times a bigram appears, put all the tokens in a list and use `pd.Series(list).value_counts()`.*

In [23]:
# Your code here

# Exercise 3.3

You can represent a document as the weighted sum of the word embeddings of the terms it contains, where the term weights are given by, e.g. Tf-Idf.

- Compute the document embeddings for the US presidential speeches corpus.
- Write a function that, given a document (speech) id, finds the top n most similar documents (as `most_similar` does for words).


The data you need:
- `data.preprocessed`: the text of each speech (a string), to fit the TF-IDF vectorizer;
- `data.tokens`: the words of each speech (a list), to build its embedding.

The next cell is given: it prepares the TF-IDF matrix and a skip-gram model. Then complete `get_document_embedding` (replace the `___`).

Hints (but there may be other / better solutions):

- To obtain the TfIdf values from the 'tfidf' matrix returned by fit_transform, use 'tfidf[doc_id, term_index]'. The list of terms (features) is returned by 'vectorizer.get_feature_names_out()'
- To obtain the Word2Vec embedding for a word, use 'model.wv[word]', where 'model = Word2Vec(...)'
- To obtain the document embedding, iterate over all of its words (that are present both as features for the TfIdf vectorizer and in the Word2Vec model vocabulary), and sum over all the products: (model.wv[word] * tfidf[doc_id, word_index])

In [30]:
# Given: TF-IDF of the speeches and a skip-gram Word2Vec model (trained on the sentences of Exercise 3.1)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
docs_tfidf = vectorizer.fit_transform(data.preprocessed)
model = Word2Vec(sentences=corpus, window=5, sg=1, seed=42)

# position of each word in the TF-IDF matrix: docs_tfidf[doc_id, term_index[word]]
term_index = {word: i for i, word in enumerate(vectorizer.get_feature_names_out())}

In [31]:
# Complete the function: replace the ___
def get_document_embedding(doc_id):
    embedding = np.zeros(model.vector_size)
    for word in set(data.tokens[doc_id]):   # each word once: the TF-IDF weight already counts its repetitions
        if word in term_index and word in model.wv:
            embedding = embedding + ___ * ___
    return embedding

In [32]:
# Your code here
# 1. compute the embedding of every speech (a matrix with one row per speech)
# 2. compute the cosine similarity between all the speeches
# 3. write top_n_most_similar_document(doc_id, doc_sim, top_n=5): the year and president of the most similar speeches
#    (doc_sim = the similarity matrix: so you can reuse the function in Exercise 3.4)

# Exercise 3.4

Use Doc2Vec (https://radimrehurek.com/gensim/models/doc2vec.html) to compute the embeddings for the documents in the presidential speeches dataset. Write a function that finds the most similar documents for a given document(adapt the function of Exercise 3.3).

*Hint: see `Doc2Vec` in the Introduction to gensim. Use `data.tokens` (the words of each speech) to build one `TaggedDocument` per speech, with the row number as tag; after training, `doc_model.dv[i]` is the vector of speech `i`.*

In [39]:
# Your code here

from gensim.models.doc2vec import Doc2Vec, TaggedDocument